# ECOBITE - Exploratory Data Analysis & Machine Learning Training Pipeline

**Objective:** Build an end-to-end machine learning system to predict supermarket inventory waste risk and forecast demand before product expiry to enable dynamic, revenue-aware discount recommendations.

## Notebook Contents:
1. Data Loading & Exploratory Analysis
2. Feature Engineering & Domain Metrics
3. Strict Featurization Ordering & Preprocessing (No Data Leakage)
4. Waste Risk Classification (Random Forest vs Logistic Regression vs Decision Tree)
5. Demand Prediction Regression (Random Forest Regressor)
6. Explainability & Feature Importance Analysis
7. Conclusion & Business Impact Summary

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in python path
sys.path.append('..')
from src.feature_engineering import add_engineered_features, get_feature_columns
from src.data_preprocessing import build_preprocessor_pipeline, prepare_features_and_targets

data_path = '../data/raw/supermarket_inventory.csv'
if not os.path.exists(data_path):
    from src.data.raw.generate_synthetic_data import main as gen_data
    gen_data()

df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape}")
df.head()

## 1. Exploratory Data Analysis
Analyzing target label distribution and relationships between stock, days to expiry, and waste probability.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=df, x='Waste_Label', palette='viridis', ax=axes[0])
axes[0].set_title('Waste Label Distribution (0: Sell, 1: Unsold Waste)')

sns.scatterplot(data=df, x='Days_to_Expiry', y='Current_Stock', hue='Waste_Label', palette='Set1', ax=axes[1])
axes[1].set_title('Stock vs Days to Expiry by Waste Risk')
plt.tight_layout()
plt.show()

## 2. Feature Engineering & Preprocessing Pipeline
To prevent data leakage, we perform train-test split *before* fitting encoders or scalers.

In [ ]:
from sklearn.model_selection import train_test_split

X, y = prepare_features_and_targets(df, target_col='Waste_Label')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

preprocessor = build_preprocessor_pipeline()
X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)
print(f"Transformed Training Features Shape: {X_train_trans.shape}")

## 3. Classification Model Training & Evaluation
Comparing Random Forest Classifier against Logistic Regression and Decision Trees.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

rf_clf = RandomForestClassifier(n_estimators=150, max_depth=12, random_state=42, class_weight='balanced')
rf_clf.fit(X_train_trans, y_train)

y_pred = rf_clf.predict(X_test_trans)
y_proba = rf_clf.predict_proba(X_test_trans)[:, 1]

print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

## 4. Feature Importance Analysis
Understanding the key features driving waste risk predictions.

In [ ]:
feature_names = preprocessor.get_feature_names_out()
importances = rf_clf.feature_importances_
feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=feat_imp.values, y=feat_imp.index, palette='crest')
plt.title('Top 10 Feature Importances (ECOBITE Waste Model)')
plt.xlabel('Importance')
plt.show()

## 5. Summary & Conclusion
- The **Random Forest Classifier** achieved high Recall and F1-score, ensuring products at risk of expiring unsold are reliably identified.
- Engineered features such as **Stock Coverage Days**, **Expiry Urgency**, and **Stock-to-Demand Ratio** significantly enhanced predictive signal.
- Models are saved and deployed via the interactive Streamlit decision dashboard.